In [1]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, T5ForConditionalGeneration, T5Tokenizer
import torch
from utils.config import q_type_models, qa_models, CV_DATA
import pandas as pd
import json
import gc
import time
pd.set_option('display.max_columns', None) 
pd.set_option('max_colwidth', None) # show full width of showing cols
pd.set_option("expand_frame_repr", False) # print cols side by side as it's supposed to be
from utils.config import q_type_models, qa_models, CV_DATA
from utils.model_pipelines import extract_skills_from_text, get_qa_model_out_raw_batch, get_question_type_prediction_batch, extract_skills_from_text
import re

In [2]:
questions= [
    "What is your full name?",
    "What is your email id?",
    "What is your phone country code?",
    "What is your phone number?",
    "What is your current location?",
    "What is your preferred location?",
    "What is your notice period in days?",
    "Is your notice period negotiable?",
    "What is your current ctc?",
    "What is your expected ctc?",
]

# Using classifier

### For single input at a time

In [3]:
def get_question_type_prediction(text):
    # Load model & tokenizer inside function (so it's released later)
    QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_models[-1])
    QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_models[-1])
    ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_CLASSIFIER_MODEL.to(device)
        QUESTION_CLASSIFIER_MODEL.eval()
        inputs = QUESTION_CLASSIFIER_TOKENIZER(text, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
        logits = outputs.logits
        predicted_classes = torch.argmax(logits, dim=1)
        result = ID2LABEL[predicted_classes.item()]
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider using a smaller batch size or switching to CPU.")
    finally:
        # Cleanup: Release memory after execution
        del QUESTION_CLASSIFIER_MODEL, QUESTION_CLASSIFIER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()
    return result
# get_question_type_prediction(questions[-1])

### For multiple input at a time/ batch processing

In [4]:
def get_question_type_predictions(texts, batch_size= 64):
    # Load model & tokenizer inside function (so it's released later)
    QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_models[-1])
    QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_models[-1])
    ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label
    results = []
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_CLASSIFIER_MODEL.to(device)
        QUESTION_CLASSIFIER_MODEL.eval()
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = QUESTION_CLASSIFIER_TOKENIZER(batch_texts, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
            logits = outputs.logits
            predicted_classes = torch.argmax(logits, dim=1)
            batch_results = [ID2LABEL[idx.item()] for idx in predicted_classes]
            results.extend(batch_results)
            del inputs, outputs, logits
            torch.cuda.empty_cache()
            gc.collect()
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider reducing batch size further or using CPU.")
    except Exception as e:
        print(f"Error occurred: {e}")
    finally:
        del QUESTION_CLASSIFIER_MODEL, QUESTION_CLASSIFIER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()
    return results

# Example usage
# get_question_type_predictions(questions)

### Comparing results for all different optimiser result:
- multiple input for all the avilable classifier

In [5]:
def get_dataframe_for_comparision_question_classifier(questions, batch_size=64):
    table_data = {"questions": questions}
    for q_type_model in q_type_models:
        # Load model & tokenizer inside loop (so they are released after each iteration)
        QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_model)
        QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_model)
        ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label
        predictions = []
        try:
            torch.cuda.empty_cache()
            gc.collect()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            QUESTION_CLASSIFIER_MODEL.to(device)
            QUESTION_CLASSIFIER_MODEL.eval()
            for i in range(0, len(questions), batch_size):
                batch_questions = questions[i:i + batch_size]
                inputs = QUESTION_CLASSIFIER_TOKENIZER(batch_questions, padding=True, truncation=True, return_tensors="pt").to(device)
                with torch.no_grad():
                    outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
                logits = outputs.logits
                predicted_classes = torch.argmax(logits, dim=1)
                batch_predictions = [ID2LABEL[idx.item()] for idx in predicted_classes]
                predictions.extend(batch_predictions)
                del inputs, outputs, logits
                torch.cuda.empty_cache()
                gc.collect()
        except torch.cuda.OutOfMemoryError:
            print(f"CUDA Out of Memory for model {q_type_model}! Consider reducing batch size further or switching to CPU.")
            predictions = None
        except Exception as e:
            print(f"Error occurred with model {q_type_model}: {e}")
            predictions = None
        finally:
            del QUESTION_CLASSIFIER_MODEL, QUESTION_CLASSIFIER_TOKENIZER
            torch.cuda.empty_cache()
            gc.collect()
        table_data[q_type_model.split("-")[-1]] = predictions
    return pd.DataFrame(table_data)


# Example usage
# get_dataframe_for_comparision_question_classifier(questions)

# Question answer model

### For single input at a time

In [6]:
MAX_TOKEN_SIZE= 2048
def get_qa_model_out_raw(question):
    # Load model & tokenizer inside function (so they are released after execution)
    QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_models[-1])
    QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_models[-1], legacy=True)
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_ANSWER_MODEL.to(device)
        QUESTION_ANSWER_MODEL.eval()
        # Get predicted question type
        predicted_question_type = get_question_type_prediction(question)
        skill_list= extract_skills_from_text(question) if  predicted_question_type== "skills" else []
        context= [line for line in CV_DATA["skills"].split("\n") if any(re.search(rf'\b{skill}\b', line, re.IGNORECASE) for skill in skill_list)] if skill_list else CV_DATA.get(predicted_question_type, "")
        input_text = f"You are a candidate filling job application form answer the question based on the given information.\nQuestion: {question}\ncontext: {context}"
        inputs = QUESTION_ANSWER_TOKENIZER(input_text, return_tensors="pt", max_length=1024, truncation=True).to(device)
        output = QUESTION_ANSWER_MODEL.generate(**inputs, max_length=50)
        return predicted_question_type, QUESTION_ANSWER_TOKENIZER.decode(output[0], skip_special_tokens=True)
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider reducing input size or using CPU.")
        output_text = None
    finally:
        # Cleanup: Release memory after execution
        del QUESTION_ANSWER_MODEL, QUESTION_ANSWER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()
    return predicted_question_type, output_text


# Example usage
# get_qa_model_out_raw("How would you rate yourself on Python coding skills in the scale of 10?")

### For multiple input at a time/ batch processing

In [7]:
"""
    For my configuration nvidia 1650Ti 4GB graphic batch size=4 is the max my system can procede
"""
def get_qa_model_out_batch(questions, batch_size=4):
    QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_models[-1])
    QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_models[-1], legacy=False)
    inputs = None
    decoded_outputs = []
    predicted_types = []
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_ANSWER_MODEL.to(device)
        QUESTION_ANSWER_MODEL.eval()
        for i in range(0, len(questions), batch_size):
            batch_questions = questions[i:i + batch_size]
            batch_predicted_types = [get_question_type_prediction(q) for q in batch_questions]
            batch_contexts = [CV_DATA.get(p_type, "") for p_type in batch_predicted_types]
            batch_input_texts = [f"You are a candidate filling job application form answer the question based on the given information.\nQuestion: {q}\ncontext: {c}" for q, c in zip(batch_questions, batch_contexts)]
            inputs = QUESTION_ANSWER_TOKENIZER(batch_input_texts, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                batch_outputs = QUESTION_ANSWER_MODEL.generate(input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)
            batch_decoded_outputs = [QUESTION_ANSWER_TOKENIZER.decode(out, skip_special_tokens=True) for out in batch_outputs]
            decoded_outputs.extend(batch_decoded_outputs)
            predicted_types.extend(batch_predicted_types)
            del inputs, batch_outputs, batch_input_texts
            torch.cuda.empty_cache()
            gc.collect()
        if not decoded_outputs:
            decoded_outputs = [""] * len(questions)
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider reducing batch size further or using CPU.")
        decoded_outputs = [""] * len(questions)
    except Exception as e:
        print(f"Error occurred: {e}")
        decoded_outputs = [""] * len(questions)
    finally:
        del QUESTION_ANSWER_MODEL, QUESTION_ANSWER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()

    # Return results as a DataFrame
    result_df = pd.DataFrame({
        'Question': questions,
        'Question Type': predicted_types,
        'Answer': decoded_outputs
    })
    return result_df

# Example usage
get_qa_model_out_batch(questions)

,Question,Question Type,Answer
0,What is your full name?,personal_information,Manab Boro
1,What is your email id?,personal_information,3
2,What is your phone country code?,personal_information,871454
3,What is your phone number?,personal_information,9101925089
4,What is your current location?,personal_information,In India
5,What is your preferred location?,personal_information,
6,What is your notice period in days?,availability,30 days
7,Is your notice period negotiable?,availability,No
8,What is your current ctc?,current_ctc,517000
9,What is your expected ctc?,expected_ctc,850000


### Comparing results for all different optimiser result:
- multiple input for all the avilable qa models

In [8]:
def get_dataframe_for_comparision_question_answer(questions, batch_size=4):
    model_index= 0
    for qa_type_model in qa_models:
        model_index+= 1
        QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_type_model)
        QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_type_model, legacy=False)
        inputs = None
        decoded_outputs = []
        predicted_types = []
        try:
            torch.cuda.empty_cache()
            gc.collect()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            QUESTION_ANSWER_MODEL.to(device)
            QUESTION_ANSWER_MODEL.eval()
            for i in range(0, len(questions), batch_size):
                batch_questions = questions[i:i + batch_size]
                batch_predicted_types = [get_question_type_prediction(q) for q in batch_questions]
                batch_contexts = [CV_DATA.get(p_type, "") for p_type in batch_predicted_types]
                batch_input_texts = [f"question: {q} context: {c}" for q, c in zip(batch_questions, batch_contexts)]
                inputs = QUESTION_ANSWER_TOKENIZER(batch_input_texts, padding=True, truncation=True, return_tensors="pt").to(device)
                with torch.no_grad():
                    batch_outputs = QUESTION_ANSWER_MODEL.generate(input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)
                batch_decoded_outputs = [QUESTION_ANSWER_TOKENIZER.decode(out, skip_special_tokens=True) for out in batch_outputs]
                decoded_outputs.extend(batch_decoded_outputs)
                predicted_types.extend(batch_predicted_types)
                del inputs, batch_outputs, batch_input_texts
                torch.cuda.empty_cache()
                gc.collect()
            if not decoded_outputs:
                decoded_outputs = [""] * len(questions)
        except torch.cuda.OutOfMemoryError:
            print("CUDA Out of Memory! Consider reducing batch size further or using CPU.")
            decoded_outputs = [""] * len(questions)
        except Exception as e:
            print(f"Error occurred: {e}")
            decoded_outputs = [""] * len(questions)
        finally:
            del QUESTION_ANSWER_MODEL, QUESTION_ANSWER_TOKENIZER
            torch.cuda.empty_cache()
            gc.collect()
        data[f"model_out_{data}"]= decoded_outputs
    return pd.DataFrame(data)



# get_dataframe_for_comparision_question_answer(questions)